In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [3]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/2024-11/fdt/*'))
files_fdt

['/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T001503_V202602220112_0451090501.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T031503_V202602220112_0451090502.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T061503_V202602220112_0451090503.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T091503_V202602220112_0451090504.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T121503_V202602220112_0451090505.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T151503_V202602220112_0451090506.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T181503_V202602220112_0451090507.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241109T211503_V202602220112_0451090508.fits.gz',
 '/home/ulyanov/data/cross/2024-11/fdt/solo_L2_phi-fdt-blos_20241110T001503_V202602220112_0451100501.fits.gz',
 

In [7]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/2024-11/hmi/*'))
files_hmi

['/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_031630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_061630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_091630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_121630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_151630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_181630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241109_211630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241110_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241110_031630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/2024-11/hmi/hmi.M_45s.20241110_061630_TAI.2.magnetogram.fits',
 '/home/ul

In [5]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xu, yu = s['xu'], s['yu']

In [20]:
file_hmi = files_hmi[-1]
file_fdt = files_fdt[-1]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_hmi = rebin(data_hmi, 10, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1,
                     xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                     crota=header_fdt['CROTA'] + 0.25,
                     grid=crop_grid(xu, yu, header_fdt),
                     kind='bilinear')

data_fdt[np.isnan(data_hmi)] = np.nan

In [21]:
xx, yy = [0], [0]

for q in np.arange(2., 30.25, 0.5):
    w = np.sqrt(data_hmi ** 2 + data_fdt ** 2)
    t = np.where(w < q ** 2)
    x, y = data_hmi[t], data_fdt[t]
    w = 1#w[t]

    A = np.array([[np.mean(w * x ** 2), np.mean(w * x * y)],
                  [np.mean(w * x * y), np.mean(w * y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

print(u / v)


xx = np.array(xx)
yy = np.array(yy)

1.2258866288539458


In [22]:
plt.figure(figsize=(10,10))
plt.plot([-400,400], [-400,400], color='black', lw=0.5)
plt.scatter(data_hmi, data_fdt, s=0.05)
plt.plot([-u,u],[-v,v], '--', color='black', lw=1.5)
#plt.plot(xx, yy, '--', color='black', lw=1)
plt.plot()

plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-400,400)
plt.ylim(-400,400)



plt.grid(True)
plt.tight_layout()

In [23]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()

In [24]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()

In [25]:
view_fdt = View.from_header(header_fdt)
view_hmi = View.from_header(header_hmi)

u = view_fdt.to_carrington(origin='heliographic')((0,0,1))[0]
v = view_hmi.to_carrington(origin='heliographic')((0,0,1))[0]

np.arccos(np.sum(np.array(u) * np.array(v))) * 180 / np.pi

np.float64(6.314737401844805)

In [23]:
header_fdt['CRLN_OBS'], header_hmi['CRLN_OBS']

(217.04348, 218.31459)

In [24]:
header_fdt['HGLN_OBS']

-1.2256609